In [ ]:
import mne
import os
import numpy as np

# Replace with your EDF file path
username = os.getlogin()
edf_file = f'C:\\Users\\{username}\\Box\\Research_SPORTS_VR_AI\\Creative Inquiry F24\\EDF Data\\Women\\OnField\\Jenna_onfield_trial1_raw.edf'

# Load the EDF file with annotations
raw = mne.io.read_raw_edf(edf_file, preload=True)

# Extract events and event IDs from annotations
events, event_id = mne.events_from_annotations(raw)

# Define epoch parameters
tmin = -0.2  # start of epoch relative to event
tmax = 0.6   # end of epoch relative to event
baseline = (None, 0)  # optional: baseline correction

# Create epochs from raw data
epochs = mne.Epochs(raw, events, event_id, tmin=tmin, tmax=tmax,
                    baseline=baseline, preload=True)

# Output directory
output_dir = 'epochs_as_edf'
os.makedirs(output_dir, exist_ok=True)

sfreq = raw.info['sfreq']

print(f"Processing {len(epochs)} epochs...")
print("Preserving original annotation timings within each epoch.\n")

# Process each epoch individually
for i in range(len(epochs)):
    # Extract epoch data and info
    single_epoch = epochs[i]
    data = single_epoch.get_data().squeeze()
    info = single_epoch.info

    # Ensure 2D data shape
    if data.ndim == 1:
        data = data.reshape(1, -1)

    # Get absolute event time for this epoch
    event_sample = epochs.events[i, 0]
    event_time = event_sample / sfreq  # in seconds
    epoch_start_time = event_time + tmin
    epoch_end_time = event_time + tmax

    # Filter annotations that fall within this epoch window
    anns_in_epoch = []
    for onset, duration, description in zip(raw.annotations.onset,
                                            raw.annotations.duration,
                                            raw.annotations.description):
        if epoch_start_time <= onset <= epoch_end_time:
            # Calculate relative onset time within the epoch
            relative_onset = onset - epoch_start_time
            anns_in_epoch.append((relative_onset, duration, description))

    # Create a RawArray for this epoch
    raw_epoch = mne.io.RawArray(data, info)

    # Add annotations to RawArray
    if anns_in_epoch:
        ann_onsets, ann_durations, ann_descs = zip(*anns_in_epoch)
        annotations = mne.Annotations(onset=ann_onsets,
                                      duration=ann_durations,
                                      description=ann_descs)
        raw_epoch.set_annotations(annotations)
    else:
        raw_epoch.set_annotations(mne.Annotations([], [], []))


    # Save the epoch to EDF
    filename = f'epoch_{i+1:03d}.edf'
    output_path = os.path.join(output_dir, filename)

    try:
        mne.export.export_raw(output_path, raw_epoch, overwrite=True)
        print(f"Epoch {i+1}: Saved with {len(anns_in_epoch)} annotation(s)")
    except Exception as e:
        print(f"Error saving epoch {i+1}: {e}")

print(f"\n✓ Complete! Saved {len(epochs)} epochs to '{output_dir}'")
print("Each epoch includes only annotations that occurred within its time window.\n")

# Optional: Quick verification of the first few epochs
print("=== VERIFICATION (first 3 epochs) ===")
for i in range(min(3, len(epochs))):
    test_file = os.path.join(output_dir, f'epoch_{i+1:03d}.edf')
    if os.path.exists(test_file):
        test_raw = mne.io.read_raw_edf(test_file, preload=False, verbose='ERROR')
        test_anns = test_raw.annotations

        print(f"\nepoch_{i+1:03d}.edf:")
        if len(test_anns) == 0:
            print("  ⚠ No annotations found")
        else:
            for ann in test_anns:
                desc = ann['description']
                onset = ann['onset']
                print(f"  - '{desc}' at {onset:.3f}s")


Extracting EDF parameters from C:\Users\jackr\Box\Research_SPORTS_VR_AI\Creative Inquiry F24\EDF Data\Women\OnField\Jenna_onfield_trial1_raw.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 17999  =      0.000 ...    59.997 secs...
Used Annotations descriptions: [np.str_(' Pass ')]
Not setting metadata
5 matching events found
Setting baseline interval to [-0.2, 0.0] sec
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 5 events and 241 original time points ...
0 bad epochs dropped
Processing 5 epochs...
Preserving original annotation timings within each epoch.

Creating RawArray with float64 data, n_channels=30, n_times=241
    Range : 0 ... 240 =      0.000 ...     0.800 secs
Ready.
Overwriting existing file.
Epoch 1: Saved with 1 annotation(s)
Creating RawArray with float64 data, n_channels=30, n_times=241
    Range : 0 ... 240 =      0.000 ...     0.800 secs
Ready.
Overwriti

C:\Users\jackr\AppData\Local\Temp\ipykernel_25684\93797796.py:79: RuntimeWarning: EDF format requires equal-length data blocks, so 0.19666666666666666 seconds of zeros were appended to all channels when writing the final block.
  mne.export.export_raw(output_path, raw_epoch, overwrite=True)
C:\Users\jackr\AppData\Local\Temp\ipykernel_25684\93797796.py:79: RuntimeWarning: EDF format requires equal-length data blocks, so 0.19666666666666666 seconds of zeros were appended to all channels when writing the final block.
  mne.export.export_raw(output_path, raw_epoch, overwrite=True)
C:\Users\jackr\AppData\Local\Temp\ipykernel_25684\93797796.py:79: RuntimeWarning: EDF format requires equal-length data blocks, so 0.19666666666666666 seconds of zeros were appended to all channels when writing the final block.
  mne.export.export_raw(output_path, raw_epoch, overwrite=True)
C:\Users\jackr\AppData\Local\Temp\ipykernel_25684\93797796.py:79: RuntimeWarning: EDF format requires equal-length data bloc